# Risk-Adjusted Decision Making: When Expected Value Is Not Enough

**Part 3 of the Decision Theory Series**

This notebook demonstrates when pure Expected Value can mislead and how to incorporate CVaR, tail risk, and survivability into senior decision-making.

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')

from src.simulation import simulate_risk_adjusted_ab_test
from src.visualization import plot_risk_posterior

print("✅ Libraries loaded")

✅ Libraries loaded


## 1. Run Two Scenarios

In [2]:
repeatable = simulate_risk_adjusted_ab_test(is_one_shot=False, seed=42)
one_shot   = simulate_risk_adjusted_ab_test(is_one_shot=True,  seed=42)

print("Simulations completed")

Simulations completed


## 2. Risk Posterior Visualisation (One-Shot Case)

In [3]:
plot_risk_posterior(
    revenue_samples=one_shot["posterior"]["revenue_samples"],
    ev_revenue=one_shot["posterior"]["ev_revenue_gain"],
    cvar_5=one_shot["posterior"]["cvar_5"],
    var_5=one_shot["posterior"]["var_5"],
    p_ruin=one_shot["posterior"]["p_ruin"],
    save_path="../results/risk_posterior.png"
)
plt.show()

Risk posterior plot saved -> ../results/risk_posterior.png


## 3. Key Risk Metrics Comparison

In [4]:
def print_metrics(name, res):
    p = res["posterior"]
    print(f"\n{name} scenario:")
    print(f"  EV revenue gain      : ${p['ev_revenue_gain']:,.0f}")
    print(f"  5% VaR               : ${p['var_5']:,.0f}")
    print(f"  5% CVaR              : ${p['cvar_5']:,.0f}")
    print(f"  P(ruin)              : {p['p_ruin']:.1%}")

print_metrics("Repeatable (small tests)", repeatable)
print_metrics("One-shot (high-stakes launch)", one_shot)


Repeatable (small tests) scenario:
  EV revenue gain      : $4,026
  5% VaR               : $2,055
  5% CVaR              : $1,569
  P(ruin)              : 0.0%

One-shot (high-stakes launch) scenario:
  EV revenue gain      : $-4,384
  5% VaR               : $-7,535
  5% CVaR              : $-8,386
  P(ruin)              : 99.2%


## 4. Decision Framework

In [8]:
# Decision Framework - clean print with your current numbers
print("""\nDecision Table (One-Shot High-Stakes Case)
──────────────────────────────────────
Metric                    Value              Interpretation
──────────────────────    ─────────────      ─────────────────────────────
EV revenue gain           ${:,}             Materially positive on average
5% VaR                    ${:,}             Limited downside
5% CVaR                   ${:,}             Tail loss is concerning
P(ruin)                   {:.1%}             Non-negligible probability of loss
Recommendation            → Do NOT ship      CVaR and ruin risk outweigh EV
""".format(
    int(one_shot["posterior"]["ev_revenue_gain"]),
    int(one_shot["posterior"]["var_5"]),
    int(one_shot["posterior"]["cvar_5"]),
    one_shot["posterior"]["p_ruin"]
))


Decision Table (One-Shot High-Stakes Case)
──────────────────────────────────────
Metric                    Value              Interpretation
──────────────────────    ─────────────      ─────────────────────────────
EV revenue gain           $-4,383             Materially positive on average
5% VaR                    $-7,534             Limited downside
5% CVaR                   $-8,385             Tail loss is concerning
P(ruin)                   99.2%             Non-negligible probability of loss
Recommendation            → Do NOT ship      CVaR and ruin risk outweigh EV

